# ML Network Intrusion Detector

**Student:** Amal Johnson

This Python/Jupyter Notebook implements the IBM SkillsBuild project topic **ML Network Intrusion Detector**.

The system uses the public NSL-KDD dataset and classifies network connections as **Normal** or **Attack** using Scikit-learn.


## 1. Import Libraries

Run the cells in order from top to bottom. The notebook is designed for Google Colab.


In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

print("Libraries imported successfully.")


## 2. Download the NSL-KDD Dataset

The notebook first checks whether the files already exist. If not, it tries reliable public mirrors.

This avoids continuing with an empty or incorrectly downloaded file.


In [ ]:
TRAIN_FILE = "KDDTrain+.txt"
TEST_FILE = "KDDTest+.txt"

TRAIN_URLS = [
    "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt",
    "https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTrain%2B.txt"
]

TEST_URLS = [
    "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest%2B.txt",
    "https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTest%2B.txt"
]

def download_with_fallback(urls, filename):
    if os.path.exists(filename) and os.path.getsize(filename) > 1000:
        print(f"{filename} already exists.")
        return

    last_error = None

    for url in urls:
        try:
            print(f"Trying: {url}")
            urllib.request.urlretrieve(url, filename)

            if os.path.getsize(filename) > 1000:
                print(f"Downloaded successfully: {filename}")
                return

        except Exception as error:
            last_error = error
            if os.path.exists(filename):
                os.remove(filename)

    raise RuntimeError(
        f"Could not download {filename}. Check your internet connection and try again. "
        f"Last error: {last_error}"
    )

download_with_fallback(TRAIN_URLS, TRAIN_FILE)
download_with_fallback(TEST_URLS, TEST_FILE)

print("Dataset files are ready.")


## 3. Define the NSL-KDD Columns

The standard NSL-KDD text files contain 43 columns:

- 41 network features
- 1 attack label
- 1 difficulty value


In [ ]:
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty"
]

train_df = pd.read_csv(
    TRAIN_FILE,
    header=None,
    names=columns,
    skipinitialspace=True
)

test_df = pd.read_csv(
    TEST_FILE,
    header=None,
    names=columns,
    skipinitialspace=True
)

print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)

if train_df.shape[1] != 43 or test_df.shape[1] != 43:
    raise ValueError("Dataset format is incorrect. Expected 43 columns.")

print("\nDataset loaded correctly.")
train_df.head()


## 4. Prepare the Target

Every `normal` record becomes **0 (Normal)**.

Every other NSL-KDD label becomes **1 (Attack)**.

The original label and difficulty columns are removed from the model inputs to prevent target leakage.


In [ ]:
def make_binary_target(df):
    df = df.copy()

    labels = (
        df["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.rstrip(".")
    )

    df["target"] = (labels != "normal").astype(int)
    return df

train_df = make_binary_target(train_df)
test_df = make_binary_target(test_df)

X_train = train_df.drop(columns=["label", "difficulty", "target"])
y_train = train_df["target"]

X_test = test_df.drop(columns=["label", "difficulty", "target"])
y_test = test_df["target"]

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining class counts:")
print(
    y_train.value_counts()
    .sort_index()
    .rename(index={0: "Normal", 1: "Attack"})
)

print("\nTesting class counts:")
print(
    y_test.value_counts()
    .sort_index()
    .rename(index={0: "Normal", 1: "Attack"})
)


## 5. Class Distribution

A simple Matplotlib bar chart is used here to avoid plotting errors caused by mismatched category and value lengths.


In [ ]:
normal_count = int((y_train == 0).sum())
attack_count = int((y_train == 1).sum())

labels = ["Normal", "Attack"]
values = [normal_count, attack_count]

plt.figure(figsize=(7, 4))
plt.bar(labels, values)
plt.title("Training Data Class Distribution")
plt.xlabel("Traffic Class")
plt.ylabel("Number of Connections")
plt.tight_layout()
plt.show()


## 6. Define Preprocessing

The NSL-KDD data contains both categorical and numerical features.

- Categorical features are imputed and one-hot encoded.
- Numerical features are imputed and standardized.
- Unknown categories in the test set are safely ignored.


In [ ]:
categorical_features = ["protocol_type", "service", "flag"]
numeric_features = [c for c in X_train.columns if c not in categorical_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Categorical features:", categorical_features)
print("Number of numeric features:", len(numeric_features))
print("Preprocessing pipeline created successfully.")


## 7. Create the Machine-Learning Models

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1500,
        solver="liblinear",
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

print("Logistic Regression and Random Forest models created.")


## 8. Train the Models

In [ ]:
print("Training Logistic Regression...")
logistic_model.fit(X_train, y_train)

print("Training Random Forest...")
random_forest_model.fit(X_train, y_train)

print("Training completed successfully.")


## 9. Evaluate the Models

Accuracy, precision, recall and F1-score are reported for both models.


In [ ]:
def evaluate_model(model, name):
    predictions = model.predict(X_test)

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0)
    }

    print("\n" + name)
    print("-" * len(name))
    print(f"Accuracy : {metrics['Accuracy']:.4f}")
    print(f"Precision: {metrics['Precision']:.4f}")
    print(f"Recall   : {metrics['Recall']:.4f}")
    print(f"F1-score : {metrics['F1-score']:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            target_names=["Normal", "Attack"],
            zero_division=0
        )
    )

    return predictions, metrics

logistic_predictions, logistic_metrics = evaluate_model(
    logistic_model,
    "Logistic Regression"
)

rf_predictions, rf_metrics = evaluate_model(
    random_forest_model,
    "Random Forest"
)

results = pd.DataFrame([logistic_metrics, rf_metrics])

print("\nFinal comparison:")
display(results)


## 10. Compare Model Performance

In [ ]:
metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]

ax = results.set_index("Model")[metric_names].plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Network Intrusion Detection Model Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Confusion Matrix

The confusion matrix shows how correctly the Random Forest model classified Normal and Attack connections.


In [ ]:
cm = confusion_matrix(y_test, rf_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Attack"],
    yticklabels=["Normal", "Attack"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()


## 12. Random Forest Feature Importance

The trained Random Forest is used to identify the most influential transformed input features.


In [ ]:
rf_preprocessor = random_forest_model.named_steps["preprocessor"]
rf_classifier = random_forest_model.named_steps["classifier"]

feature_names = rf_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_classifier.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 20 important features:")
display(importance_df.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=importance_df.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Top 15 Network Features")
plt.tight_layout()
plt.show()


## 13. Save the Models and Results

In [ ]:
joblib.dump(logistic_model, "logistic_intrusion_detector.joblib")
joblib.dump(random_forest_model, "random_forest_intrusion_detector.joblib")

results.to_csv("model_results.csv", index=False)

print("Files saved successfully:")
print("1. logistic_intrusion_detector.joblib")
print("2. random_forest_intrusion_detector.joblib")
print("3. model_results.csv")


## 14. Sample Prediction

In [ ]:
loaded_model = joblib.load("random_forest_intrusion_detector.joblib")

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]
probability = loaded_model.predict_proba(sample)[0][1]

print("Predicted class:", "Attack" if prediction == 1 else "Normal")
print(f"Attack probability: {probability:.2%}")


## 15. Final Conclusion

The project successfully demonstrates a Scikit-learn based machine-learning workflow for network intrusion detection.

It includes:

- public cybersecurity data
- data validation and cleaning
- binary classification
- Logistic Regression
- Random Forest
- accuracy, precision, recall and F1-score
- confusion matrix
- feature importance
- saved machine-learning models
- sample prediction

This is an educational cybersecurity project and is not intended to replace a production intrusion-detection system.
